1.Install VADER

- pip install vader-sentiment nltk <br>
- Download VADER lexicon

In [72]:
import pandas as pd
import re
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon', quiet=True)

True

2.Load Dataset

- Read restaurants_combined_reviews_final.csv <br>
- Check shape and columns

In [73]:
def load_data(filepath):
    """Step 1: Load dataset"""
    df = pd.read_csv(filepath)
    print(f"Loaded dataset: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    return df

3.Data Exploration

- Look at sample reviews
- Check review_rating distribution
- Check for null values in review_tex

In [74]:
def explore_data(df, text_col='review_text', rating_col='review_rating'):
    """Step 2: Data Exploration"""
    print("\nSample Reviews:")
    print(df[[text_col, rating_col]].head(5).to_string())

    print(f"\nRating Distribution:")
    print(df[rating_col].value_counts().sort_index())

    print(f"\nNull values in '{text_col}': {df[text_col].isnull().sum()}")
    print(f"Duplicates: {df[text_col].duplicated().sum()}")
    print(f"Very short reviews (< 3 words): {df[text_col].str.split().str.len().lt(3).sum()}")
    print(f"Whitespace-only: {df[text_col].str.strip().eq('').sum()}")
    return df

In [75]:
def clean_data(df, text_col='review_text'):
    """Step 3: Data Cleaning — remove null/empty/duplicate rows"""
    before = len(df)
    df = df[df[text_col].notna()].copy()
    df = df[df[text_col].str.strip() != '']
    df = df.drop_duplicates(subset=text_col, keep='first')
    print(f"\nRemoved {before - len(df)} rows | Remaining: {len(df)}")
    return df

In [76]:
def preprocess_text(text):
    """Helper: Clean individual review text"""
    text = str(text).strip()
    if text.lower() in ['null', 'none', 'nan', 'n/a', '']:
        return ''
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_data(df, text_col='review_text'):
    """Step 4: Text Preprocessing"""
    df['review_text_clean'] = df[text_col].apply(preprocess_text)

    empty = df['review_text_clean'].str.strip().eq('').sum()
    if empty > 0:
        df = df[df['review_text_clean'].str.strip() != '']
        print(f"Removed {empty} string-null reviews after preprocessing")

    print(f"\nPreprocessing complete | Total reviews: {len(df)}")
    print(f"Sample: {df['review_text_clean'].iloc[0][:100]}...")
    return df

In [77]:
def vader_sentiment(df, text_col='review_text_clean'):
    """Step 5: VADER Sentiment Scoring"""
    sia = SentimentIntensityAnalyzer()

    df['vader_scores']    = df[text_col].apply(lambda x: sia.polarity_scores(x))
    df['vader_neg']       = df['vader_scores'].apply(lambda x: x['neg'])
    df['vader_neu']       = df['vader_scores'].apply(lambda x: x['neu'])
    df['vader_pos']       = df['vader_scores'].apply(lambda x: x['pos'])
    df['vader_compound']  = df['vader_scores'].apply(lambda x: x['compound'])

    df['vader_sentiment'] = df['vader_compound'].apply(
        lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
    )

    print(f"\nVADER Scoring complete!")
    print(f"\nSentiment Distribution:")
    print(df['vader_sentiment'].value_counts())
    return df

In [78]:
def run_pipeline(filepath, text_col='review_text', rating_col='review_rating'):
    """Master pipeline — runs all steps in order"""
    df = load_data(filepath)
    df = explore_data(df, text_col, rating_col)
    df = clean_data(df, text_col)
    df = preprocess_data(df, text_col)
    df = vader_sentiment(df, text_col='review_text_clean')
    return df

# RUN
df = run_pipeline('Dataset_prj/restaurants_combined_reviews_final.csv')

Loaded dataset: (1173, 12)
Columns: ['place_id', 'restaurant_name', 'address', 'latitude', 'longitude', 'restaurant_rating', 'user_rating_count', 'review_text', 'review_rating', 'review_time', 'author_name', 'area']

Sample Reviews:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

## Step 7: Compare with Actual Ratings

- Map review_rating to sentiment (≥4=Positive, ≤2=Negative)
- Check if VADER predictions match actual ratings

In [79]:
def map_rating_to_sentiment(rating):
    """Map actual rating to sentiment"""
    if rating >= 4:
        return 'Positive'
    elif rating <= 2:
        return 'Negative'
    else:
        return 'Neutral'

df['actual_sentiment'] = df['review_rating'].apply(map_rating_to_sentiment)

print("Actual Sentiment Distribution (from ratings):")
print(df['actual_sentiment'].value_counts())

print(f"\n\nComparison: VADER vs Actual")
print("="*50)
comparison_df = pd.DataFrame({
    'VADER Prediction': df['vader_sentiment'].value_counts(),
    'Actual Sentiment': df['actual_sentiment'].value_counts()
})
print(comparison_df)

Actual Sentiment Distribution (from ratings):
actual_sentiment
Positive    864
Negative    144
Neutral      79
Name: count, dtype: int64


Comparison: VADER vs Actual
          VADER Prediction  Actual Sentiment
Negative               NaN             144.0
Neutral                NaN              79.0
Positive               NaN             864.0
negative              92.0               NaN
neutral               24.0               NaN
positive             971.0               NaN


## Step 8: Evaluate Performance

- Calculate Accuracy, Precision, Recall, F1-score
- Create confusion matrix

In [80]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Calculate metrics
accuracy = accuracy_score(df['actual_sentiment'], df['vader_sentiment'])
precision = precision_score(df['actual_sentiment'], df['vader_sentiment'], average='weighted', zero_division=0)
recall = recall_score(df['actual_sentiment'], df['vader_sentiment'], average='weighted', zero_division=0)
f1 = f1_score(df['actual_sentiment'], df['vader_sentiment'], average='weighted', zero_division=0)

print("VADER Performance Metrics:")
print("="*50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\n\nDetailed Classification Report:")
print(classification_report(df['actual_sentiment'], df['vader_sentiment']))

print("\nConfusion Matrix:")
cm = confusion_matrix(df['actual_sentiment'], df['vader_sentiment'], labels=['Positive', 'Negative', 'Neutral'])
print(cm)

VADER Performance Metrics:
Accuracy:  0.0000
Precision: 0.0000
Recall:    0.0000
F1 Score:  0.0000


Detailed Classification Report:
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00     144.0
     Neutral       0.00      0.00      0.00      79.0
    Positive       0.00      0.00      0.00     864.0
    negative       0.00      0.00      0.00       0.0
     neutral       0.00      0.00      0.00       0.0
    positive       0.00      0.00      0.00       0.0

    accuracy                           0.00    1087.0
   macro avg       0.00      0.00      0.00    1087.0
weighted avg       0.00      0.00      0.00    1087.0


Confusion Matrix:
[[0 0 0]
 [0 0 0]
 [0 0 0]]


c:\Users\dipsh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\dipsh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\dipsh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

## Step 9: Restaurant-Level Analysis

- Group by restaurant_name
- Average VADER scores per restaurant
- Average rating per restaurant

In [81]:
restaurant_analysis = df.groupby('restaurant_name').agg({
    'review_text': 'count',
    'review_rating': 'mean',
    'vader_compound': 'mean'
}).round(3)

restaurant_analysis.columns = ['Review Count', 'Avg Rating', 'Avg VADER Score']
restaurant_analysis = restaurant_analysis.sort_values('Review Count', ascending=False)

print("Restaurant-Level Analysis (Top 10 by Review Count):")
print("="*70)
print(restaurant_analysis.head(10))

Restaurant-Level Analysis (Top 10 by Review Count):
                                               Review Count  Avg Rating  \
restaurant_name                                                           
YABA YANG                                                 9       4.556   
Friends Restaurant                                        8       4.500   
Maak Ara                                                  8       4.250   
Jimbu Thakali by Capital Grill, New Baneshwor             7       3.571   
Jheer Baneshwor                                           7       3.143   
Alcove Restaurant , a Sanctuary                           7       3.143   
House of Thakali                                          7       4.857   
Timmure Mid-Baneshwor                                     7       4.000   
Michael Grills (Mid-Baneshwor)                            6       3.833   
Parpala Bistro & Cafe                                     6       4.000   

                                               

## Step 11: Extract Key Insights

- Most positive reviews
- Most negative reviews

In [82]:
print("\nTOP 3 MOST POSITIVE REVIEWS (by VADER):")
print("="*80)
top_positive = df.nlargest(3, 'vader_compound')[['restaurant_name', 'review_text_clean', 'review_rating', 'vader_compound']]
for idx, (i, row) in enumerate(top_positive.iterrows(), 1):
    print(f"\n[{idx}] Restaurant: {row['restaurant_name']}")
    print(f"    Rating: {row['review_rating']} | VADER Score: {row['vader_compound']:.3f}")
    print(f"    Review: {row['review_text_clean'][:150]}...")

print("\n\nTOP 3 MOST NEGATIVE REVIEWS (by VADER):")
print("="*80)
top_negative = df.nsmallest(3, 'vader_compound')[['restaurant_name', 'review_text_clean', 'review_rating', 'vader_compound']]
for idx, (i, row) in enumerate(top_negative.iterrows(), 1):
    print(f"\n[{idx}] Restaurant: {row['restaurant_name']}")
    print(f"    Rating: {row['review_rating']} | VADER Score: {row['vader_compound']:.3f}")
    print(f"    Review: {row['review_text_clean'][:150]}...")


TOP 3 MOST POSITIVE REVIEWS (by VADER):

[1] Restaurant: Kathmandu Grill Restaurant & Wine Bar
    Rating: 5.0 | VADER Score: 0.998
    Review: Im honestly very selective of places to dine, but this one? nowhere more Nepali, home-like, and comforting My family and I visited on New Years Day, a...

[2] Restaurant: The Bakery Cafe Baneshwor
    Rating: 5.0 | VADER Score: 0.998
    Review: Stepping into Bakery Cafe New Baneswor is an instant mood booster. While the nice ambiance provides a comfortable and welcoming backdrop for any casua...

[3] Restaurant: Bawarchi Food Cafe (BFC)
    Rating: 5.0 | VADER Score: 0.998
    Review: I had a great experience at this restaurant. The food was absolutely delicious, fresh, and beautifully presented. Every dish had great taste and quali...


TOP 3 MOST NEGATIVE REVIEWS (by VADER):

[1] Restaurant: Durbar Thakali Grill
    Rating: 1.0 | VADER Score: -0.987
    Review: Darbar Thakali Grill Extremely Disappointing Experience Today, five of us went t

### TEXTBLOB

In [83]:
from textblob import TextBlob

def textblob_sentiment(df, text_col='review_text_clean'):
    """TextBlob Sentiment + Subjectivity"""
    df['tb_polarity']     = df[text_col].apply(lambda x: TextBlob(x).sentiment.polarity)
    df['tb_subjectivity'] = df[text_col].apply(lambda x: TextBlob(x).sentiment.subjectivity)
    df['tb_sentiment']    = df['tb_polarity'].apply(
        lambda p: 'positive' if p >= 0.05 else ('negative' if p <= -0.05 else 'neutral')
    )
    print("TextBlob Scoring Complete!")
    print(f"Avg Polarity:     {df['tb_polarity'].mean():.3f}")
    print(f"Avg Subjectivity: {df['tb_subjectivity'].mean():.3f}")
    return df

df = textblob_sentiment(df)
restaurant_features = save_results(df)

TextBlob Scoring Complete!
Avg Polarity:     0.353
Avg Subjectivity: 0.610
Shape: (218, 10)
Columns: ['restaurant_name', 'review_count', 'avg_sentiment_score', 'positive_review_ratio', 'negative_review_ratio', 'avg_review_length', 'avg_subjectivity', 'area', 'restaurant_rating', 'user_rating_count']
                        restaurant_name  review_count  avg_sentiment_score  \
0  A One Cafe Old Baneshwor, 01-4586546             5              0.79162   
1                           Aaila chhen             5              0.73028   
2                               Aalucha             5              0.44190   
3                     Aamako Bara Pasal             5              0.85036   
4       Alcove Restaurant , a Sanctuary             7              0.55070   

   positive_review_ratio  negative_review_ratio  avg_review_length  \
0               1.000000               0.000000          54.400000   
1               1.000000               0.000000          13.800000   
2               0.80

In [84]:
def save_results(df, output_path='Dataset_prj/vader_restaurant_analysis.csv'):
    """Save restaurant-level aggregated features"""
    df['review_length'] = df['review_text_clean'].apply(lambda x: len(x.split()))

    restaurant_features = df.groupby('restaurant_name').agg(
        review_count          = ('review_text_clean', 'count'),
        avg_sentiment_score   = ('vader_compound', 'mean'),
        positive_review_ratio = ('vader_sentiment', lambda x: (x == 'positive').sum() / len(x)),
        negative_review_ratio = ('vader_sentiment', lambda x: (x == 'negative').sum() / len(x)),
        avg_review_length     = ('review_length', 'mean'),
        avg_subjectivity      = ('tb_subjectivity', 'mean'),
        area                  = ('area', 'first'),
        restaurant_rating     = ('restaurant_rating', 'first'),
        user_rating_count     = ('user_rating_count', 'first'),
    ).reset_index()

    print(f"Shape: {restaurant_features.shape}")
    print(f"Columns: {restaurant_features.columns.tolist()}")
    print(restaurant_features.head())

    restaurant_features.to_csv(output_path, index=False)
    print(f"\nSaved to '{output_path}'")
    return restaurant_features

restaurant_features = save_results(df)

Shape: (218, 10)
Columns: ['restaurant_name', 'review_count', 'avg_sentiment_score', 'positive_review_ratio', 'negative_review_ratio', 'avg_review_length', 'avg_subjectivity', 'area', 'restaurant_rating', 'user_rating_count']
                        restaurant_name  review_count  avg_sentiment_score  \
0  A One Cafe Old Baneshwor, 01-4586546             5              0.79162   
1                           Aaila chhen             5              0.73028   
2                               Aalucha             5              0.44190   
3                     Aamako Bara Pasal             5              0.85036   
4       Alcove Restaurant , a Sanctuary             7              0.55070   

   positive_review_ratio  negative_review_ratio  avg_review_length  \
0               1.000000               0.000000          54.400000   
1               1.000000               0.000000          13.800000   
2               0.800000               0.200000          56.600000   
3               1.000000 

In [85]:
# Apply TextBlob sentiment analysis
df = textblob_sentiment(df, text_col='review_text_clean')

# Create and save the restaurant-level analysis dataset
restaurant_features = save_results(df, output_path='Dataset_prj/vader_restaurant_analysis.csv')
print(restaurant_features.head())

TextBlob Scoring Complete!
Avg Polarity:     0.353
Avg Subjectivity: 0.610
Shape: (218, 10)
Columns: ['restaurant_name', 'review_count', 'avg_sentiment_score', 'positive_review_ratio', 'negative_review_ratio', 'avg_review_length', 'avg_subjectivity', 'area', 'restaurant_rating', 'user_rating_count']
                        restaurant_name  review_count  avg_sentiment_score  \
0  A One Cafe Old Baneshwor, 01-4586546             5              0.79162   
1                           Aaila chhen             5              0.73028   
2                               Aalucha             5              0.44190   
3                     Aamako Bara Pasal             5              0.85036   
4       Alcove Restaurant , a Sanctuary             7              0.55070   

   positive_review_ratio  negative_review_ratio  avg_review_length  \
0               1.000000               0.000000          54.400000   
1               1.000000               0.000000          13.800000   
2               0.80